# Senbonzakura, in a browser

Abliteration removes a model's tendency to refuse. This notebook does it on a small model,
on a free GPU, with nothing installed on your machine.

It also does the part most tools skip. It measures what the edit **cost**. Removing refusal
is easy. Knowing what you broke is the work.

**Runtime, Change runtime type, GPU.** Then run the cells in order. About fifteen minutes,
most of it waiting for a download.

> Nothing here is uploaded anywhere. The model you make lives on a machine Google lends you
> and disappears when the runtime does. What you do with it is yours, and
> [the acceptable use policy](https://github.com/elementmerc/senbonzakura/blob/main/ACCEPTABLE-USE.md)
> is short enough to actually read.


## 1. Install

One command. It brings torch with it, so it takes a couple of minutes. Time enough for tea.


In [ ]:
%pip install --quiet senbonzakura
import torch
from senbonzakura import __version__
print('senbonzakura', __version__)
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')


`gpu: NONE` means the notebook will still run and will be painfully slow.
Runtime, Change runtime type, GPU, then run that cell again.


## 2. What can this model actually do?

Before changing anything, measure it. Forty arithmetic word problems, graded by whether the
final number is right. No judge model, no opinion, no vibes: correct or not.

The questions ship inside the package, so this needs no account and no download.

Write the number down. It is the only thing that makes the next step meaningful.


In [ ]:
!senbonzakura capability --model Qwen/Qwen3-1.7B --n 40 --out before.json


---

## 3. The cell that changes the model

Everything above was measurement. This edits weights.

**What it does.** It finds the direction in the model's activations that carries refusal and
takes away the model's ability to write along it. The model stops declining things.

**What it does not do.** It does not make the model cleverer, more accurate or more truthful.
It removes one behaviour, and some of what that behaviour was attached to leaves with it.
That is what the next cell measures.

**Limits, plainly.** This has never been run above 3B parameters. The instruments get worse as
models get smaller. Several numbers this project published have been withdrawn after somebody
checked them, which is why everything here ships with its controls attached.

It takes about ten minutes. Run it if you mean to.


In [ ]:
!senbonzakura \
    --model Qwen/Qwen3-1.7B \
    --track default \
    --method single-pass \
    --trials 4 \
    --capability-n 40 \
    --out ./abliterated


## 4. What did that cost?

The run already told you. This is the receipt it wrote down, which is the file anyone else
would read to check your work.

`drop` is how much arithmetic the model lost. If `headroom` says `sufficient: false`, ignore
the drop entirely: the model could not do the task before you touched it, so it cannot be
shown to have lost it.


In [ ]:
import json
record = json.load(open('abliterated/abliteration.json'))
print('refusals: ', record['baseline_refusals'], '->', record['post_bake_refusals'])
print('divergence:', round(record['post_bake_kl'], 4))
print()
print(json.dumps(record['capability'], indent=2))


## 5. Talk to it


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tok = AutoTokenizer.from_pretrained('./abliterated')
model = AutoModelForCausalLM.from_pretrained('./abliterated', dtype=torch.bfloat16).cuda()

prompt = 'Explain how a suspension bridge carries its load.'
chat = tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                               tokenize=False, add_generation_prompt=True)
out = model.generate(**tok(chat, return_tensors='pt').to('cuda'), max_new_tokens=200)
print(tok.decode(out[0], skip_special_tokens=True))


## Where to go next

- [What is and is not established](https://elementmerc.github.io/senbonzakura/guide/what-we-know):
  the running record of which of this project's claims survived being checked. Some did not.
- [How to measure a behaviour properly](https://github.com/elementmerc/senbonzakura/blob/main/METHOD.md):
  what we learned by getting numbers wrong in public.
- `senbonzakura check` reads result files from other tools and reports how their numbers could
  be wrong. No GPU, no model, no network.

If a number here does not reproduce, that is the most useful bug report this project can get.
